In [11]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)
from sklearn.preprocessing import StandardScaler

# 1. Files setup
train_files = ["c1_aggregated_features.csv", "c4_aggregated_features.csv"]
unseen_test_file = "c6_aggregated_features.csv"

flute_cols = ["flute_1", "flute_2", "flute_3"]
ignore_cols = ["cut_number", "max_wear", "cutter"] + flute_cols
smooth_window = 5

print("Step 1/4: Files load ho rahi hain...")
df_train = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
df_test = pd.read_csv(unseen_test_file).sort_values("cut_number").reset_index(drop=True)

feature_cols = [c for c in df_train.columns if c not in ignore_cols]

X_train = df_train[feature_cols]
y_train_flutes = df_train[flute_cols].to_numpy()
y_train_max = df_train["max_wear"].to_numpy() if "max_wear" in df_train.columns else y_train_flutes.max(axis=1)

X_test = df_test[feature_cols]

print(f"Step 2/4: Data ready! Features: {len(feature_cols)}, Train cuts: {len(df_train)}, Test cuts: {len(df_test)}")

# 2. Training
print("Step 3/4: Model train ho raha hai (lagbhag 10 seconds lagenge)...")
selector = SelectKBest(f_regression, k=30)
selector.fit(X_train, y_train_max)
top_features = [feature_cols[i] for i in selector.get_support(indices=True)]

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train[top_features])
X_te_scaled = scaler.transform(X_test[top_features])

ridge = Ridge(alpha=50.0)
ridge.fit(X_tr_scaled, y_train_flutes)

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    max_features=0.3,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train_flutes)

# 3. Predict & Smooth
print("Step 4/4: Predictions calculate ho rahi hain...")
p_ridge = ridge.predict(X_te_scaled)
p_rf = rf.predict(X_test)
p_hybrid = 0.5 * p_ridge + 0.5 * p_rf

p_smooth = pd.DataFrame(p_hybrid).rolling(window=smooth_window, min_periods=1).mean().to_numpy()
p_final = np.maximum.accumulate(p_smooth, axis=0)
p_max = p_final.max(axis=1)

# Results table
results = pd.DataFrame({"cut_number": df_test["cut_number"]})
for i, col in enumerate(flute_cols):
    results[f"pred_{col}"] = p_final[:, i]
results["pred_max_wear"] = p_max

print("\n================ FINAL PREDICTIONS ================")
print("\n--- Pehle 5 Cuts (Initial Tool Wear in µm) ---")
print(results.head())

print("\n--- Aakhiri 5 Cuts (End of Life Tool Wear in µm) ---")
print(results.tail())

# Metrics
y_true_flutes = df_test[flute_cols].to_numpy()
y_true_max = df_test["max_wear"].to_numpy()

mae_f = mean_absolute_error(y_true_flutes, p_final)
rmse_f = root_mean_squared_error(y_true_flutes, p_final)
r2_f = r2_score(y_true_flutes, p_final)

mae_m = mean_absolute_error(y_true_max, p_max)
rmse_m = root_mean_squared_error(y_true_max, p_max)
r2_m = r2_score(y_true_max, p_max)
mape_m = mean_absolute_percentage_error(y_true_max, p_max) * 100
acc_pct = 100.0 - mape_m

print("\n================ FINAL ACCURACY & SCORES ================")
print(f"Flutes MAE          : {mae_f:.2f} µm")
print(f"Flutes RMSE         : {rmse_f:.2f} µm")
print(f"Flutes R² Score     : {r2_f:.4f}")
print(f"--------------------------------------------------")
print(f"Max Wear MAE        : {mae_m:.2f} µm")
print(f"Max Wear RMSE       : {rmse_m:.2f} µm")
print(f"Max Wear R² Score   : {r2_m:.4f}")
print(f"Max Wear Accuracy   : {acc_pct:.2f}%")

Step 1/4: Files load ho rahi hain...
Step 2/4: Data ready! Features: 378, Train cuts: 630, Test cuts: 315
Step 3/4: Model train ho raha hai (lagbhag 10 seconds lagenge)...
Step 4/4: Predictions calculate ho rahi hain...

================ FINAL PREDICTIONS ================

--- Pehle 5 Cuts (Initial Tool Wear in µm) ---
   cut_number  pred_flute_1  pred_flute_2  pred_flute_3  pred_max_wear
0           1     58.451777     59.592563     56.551265      59.592563
1           2     58.451777     59.592563     56.551265      59.592563
2           3     58.451777     59.592563     56.551265      59.592563
3           4     58.451777     59.592563     56.551265      59.592563
4           5     58.451777     59.592563     56.551265      59.592563

--- Aakhiri 5 Cuts (End of Life Tool Wear in µm) ---
     cut_number  pred_flute_1  pred_flute_2  pred_flute_3  pred_max_wear
310         311    192.350037    209.700805    220.381822     220.381822
311         312    192.350037    209.700805    220.38